In [ ]:
from numpy._core.arrayprint import format_float_positional
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler

# **`Data Preprocessing:`**



**Load the dataset into a suitable data structure (e.g., pandas DataFrame).**


In [ ]:
df = pd.read_csv('anime.csv')

In [ ]:
df.shape

(12294, 7)

**Handle missing values, if any.**


In [ ]:
#missing values
df.isnull().sum()

,0
anime_id,0
name,0
genre,62
type,25
episodes,0
rating,230
members,0


In [ ]:
df.isnull().sum().sum()/len(df)*100

np.float64(2.5784935741011874)

In [ ]:
df.dropna(inplace=True)

In [ ]:
df.isnull().sum()

,0
anime_id,0
name,0
genre,0
type,0
episodes,0
rating,0
members,0


In [ ]:
#duplicates
df.duplicated().sum()

np.int64(0)

**Explore the dataset to understand its structure and attributes.**


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 12017 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   anime_id  12017 non-null  int64  
 1   name      12017 non-null  object 
 2   genre     12017 non-null  object 
 3   type      12017 non-null  object 
 4   episodes  12017 non-null  object 
 5   rating    12017 non-null  float64
 6   members   12017 non-null  int64  
dtypes: float64(1), int64(2), object(4)
memory usage: 751.1+ KB


In [ ]:
df.head()

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266


In [ ]:
df.describe()

,anime_id,rating,members
count,12017.000000,12017.000000,1.201700e+04
mean,13638.001165,6.478264,1.834888e+04
std,11231.076675,1.023857,5.537250e+04
min,1.000000,1.670000,1.200000e+01
25%,3391.000000,5.890000,2.250000e+02
50%,9959.000000,6.570000,1.552000e+03
75%,23729.000000,7.180000,9.588000e+03
max,34519.000000,10.000000,1.013917e+06


Explanation:
---

Loaded the anime dataset using pandas DataFrame. Checked missing values and Since the missing values percentage is less than 5%, I removed them to keep the dataset clean for further processing. Explored dataset to understand data size, data types and distributions.



# **`Feature Extraction:`**


**Decide on the features that will be used for computing similarity (e.g., genres, user ratings).**


In [ ]:
df.columns

Index(['anime_id', 'name', 'genre', 'type', 'episodes', 'rating', 'members'], dtype='object')

In [ ]:
#selecting requried features
df[['name', 'genre', 'episodes', 'rating', 'members']].head()

,name,genre,episodes,rating,members
0,Kimi no Na wa.,"Drama, Romance, School, Supernatural",1,9.37,200630
1,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",64,9.26,793665
2,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",51,9.25,114262
3,Steins;Gate,"Sci-Fi, Thriller",24,9.17,673572
4,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",51,9.16,151266


**Convert categorical features into numerical representations if necessary.**


For genre column:
---

In [ ]:
tfidf = TfidfVectorizer(stop_words='english')
genre_tfidf = tfidf.fit_transform(df['genre'])

In [ ]:
genre_tfidf.shape

(12017, 46)

For type column:
---

In [ ]:
ohe = OneHotEncoder()
type_ohe = ohe.fit_transform(df[['type']])

In [ ]:
type_ohe.shape

(12017, 6)

**Normalize numerical features if required.**



In [ ]:
num_cols = ['episodes', 'rating', 'members']

In [ ]:
#converting episodes to numeric, so that unknown becomes NaN
df['episodes'] = pd.to_numeric(df['episodes'], errors='coerce')

In [ ]:
#filling Nan with median of episodes column
df['episodes'] = df['episodes'].fillna(df['episodes'].median())

In [ ]:
min_sca = MinMaxScaler()
scaled = min_sca.fit_transform(df[num_cols])

Explanation:
---

First selected the important columns needed to calculate similarity, because these features describe each anime and help us compare them. Since genre and type are categorical columns, we converted them into numerical form. For genre, we used TF-IDF Vectorizer and for type, we used one hot encoder to convert categories. Finally, we normalized the numerical columns episodes, rating, and members using MinMaxScaler so that all values come into the same scale. This complete feature extraction makes the dataset ready for cosine similarity based recommendations.

# **`Recommendation System:`**



**Design a function to recommend anime based on cosine similarity.**


In [ ]:
genre_tfidf.shape

(12017, 46)

In [172]:
cos_sim = cosine_similarity(genre_tfidf, genre_tfidf)

In [ ]:
cos_sim.shape

(12017, 12017)

In [ ]:
indices = pd.Series(data=df.index, index=df.name)
indices

,0
name,
Kimi no Na wa.,0
Fullmetal Alchemist: Brotherhood,1
Gintama°,2
Steins;Gate,3
Gintama&#039;,4
...,...
Toushindai My Lover: Minami tai Mecha-Minami,12289
Under World,12290
Violence Gekiga David no Hoshi,12291


In [ ]:
indices.index

Index(['Kimi no Na wa.', 'Fullmetal Alchemist: Brotherhood', 'Gintama°',
       'Steins;Gate', 'Gintama&#039;',
       'Haikyuu!!: Karasuno Koukou VS Shiratorizawa Gakuen Koukou',
       'Hunter x Hunter (2011)', 'Ginga Eiyuu Densetsu',
       'Gintama Movie: Kanketsu-hen - Yorozuya yo Eien Nare',
       'Gintama&#039;: Enchousen',
       ...
       'Silent Chaser Kagami', 'Super Erotic Anime', 'Teleclub no Himitsu',
       'Tenshi no Habataki Jun', 'The Satisfaction',
       'Toushindai My Lover: Minami tai Mecha-Minami', 'Under World',
       'Violence Gekiga David no Hoshi',
       'Violence Gekiga Shin David no Hoshi: Inma Densetsu',
       'Yasuji no Pornorama: Yacchimae!!'],
      dtype='object', name='name', length=12017)

In [ ]:
def recommend_anime(anime_name, top_n=10):
  if anime_name in indices.index:
    idx = indices[anime_name]

    similar=sorted(list(enumerate(cos_sim[idx])), key=lambda x: x[1], reverse=True)

    print(f'Recommended anime for {anime_name}:')
    print('-'*30)
    for i in similar[1:top_n+1]:
      print(indices.index[i[0]])

  else:
    print("Anime is not found")

In [ ]:
recommend_anime('Naruto')

Recommended anime for Naruto:
------------------------------
Naruto: Shippuuden
Naruto
Boruto: Naruto the Movie - Naruto ga Hokage ni Natta Hi
Naruto x UT
Naruto: Shippuuden Movie 4 - The Lost Tower
Naruto: Shippuuden Movie 3 - Hi no Ishi wo Tsugu Mono
Naruto Shippuuden: Sunny Side Battle
Naruto Soyokazeden Movie: Naruto to Mashin to Mitsu no Onegai Dattebayo!!
Kyutai Panic Adventure!
Naruto: Shippuuden Movie 6 - Road to Ninja


**Given a target anime, recommend a list of similar anime based on cosine similarity scores.**

In [ ]:
recommend_anime('One Piece')

Recommended anime for One Piece:
------------------------------
One Piece: Episode of Merry - Mou Hitori no Nakama no Monogatari
One Piece: Episode of Nami - Koukaishi no Namida to Nakama no Kizuna
One Piece: Episode of Sabo - 3 Kyoudai no Kizuna Kiseki no Saikai to Uketsugareru Ishi
One Piece Film: Strong World Episode 0
One Piece: Episode of Luffy - Hand Island no Bouken
One Piece Movie 4: Dead End no Bouken
One Piece Movie 9: Episode of Chopper Plus - Fuyu ni Saku, Kiseki no Sakura
One Piece: Adventure of Nebulandia
One Piece Movie 5: Norowareta Seiken
One Piece: Umi no Heso no Daibouken-hen


**Experiment with different threshold values for similarity scores to adjust the recommendation list size.**



In [ ]:
def recommend_anime_threshold(anime_name, threshold=0.5, top_n=10):
  if anime_name in indices.index:
    idx = indices[anime_name]

    similar = sorted(list(enumerate(cos_sim[idx])), key=lambda x: x[1], reverse=True)

    print(f'Recommendations for {anime_name} (threshold = {threshold}):')
    print('-'*45)

    count = 0
    for i in similar[1:]:
      if i[1] >= threshold:
        print(indices.index[i[0]], "-> score:", round(i[1], 2))
        count += 1

      if count == top_n:
        break

    if count == 0:
      print("No similar anime found")
  else:
    print("Anime is not found")

In [ ]:
recommend_anime_threshold('Naruto', 0.4, top_n=10)

Recommendations for Naruto (threshold = 0.4):
---------------------------------------------
Naruto: Shippuuden -> score: 1.0
Naruto -> score: 1.0
Boruto: Naruto the Movie - Naruto ga Hokage ni Natta Hi -> score: 1.0
Naruto x UT -> score: 1.0
Naruto: Shippuuden Movie 4 - The Lost Tower -> score: 1.0
Naruto: Shippuuden Movie 3 - Hi no Ishi wo Tsugu Mono -> score: 1.0
Naruto Shippuuden: Sunny Side Battle -> score: 1.0
Naruto Soyokazeden Movie: Naruto to Mashin to Mitsu no Onegai Dattebayo!! -> score: 1.0
Kyutai Panic Adventure! -> score: 0.98
Naruto: Shippuuden Movie 6 - Road to Ninja -> score: 0.95


**Analyze the performance of the recommendation system and identify areas of improvement.**

Performance Analysis:
---
	•	The model gives recommendations based on genre similarity, so results are relevant for similar genre anime.
	•	If an anime has very unique genres, recommendations may be limited.
	•	Some recommendations may look less accurate because we used only genre text and not user ratings.

Areas of Improvement:
---
	•	Combine genre + rating + members for better recommendations
	•	Use hybrid recommendation system (content + collaborative filtering)
	•	Apply better text cleaning for genre column
	•	Try top_n and threshold tuning to improve recommendation quality

Explanation:
---

We implemented a content-based recommendation system using TF-IDF + cosine similarity to recommend anime based on their genre similarity. Here we followed a content-based (similarity-based) approach instead of pure collaborative filtering because our dataset mainly contains anime features like genre/type/ratings, and we don’t have a complete user–item rating matrix to learn user preference patterns; so we recommend items by comparing the content/features of anime with each other. First, we converted the genre text into numerical vectors using TF-IDF, then computed a cosine similarity matrix to measure how close two anime are (higher score means more similar). Using indices, we mapped each anime name to its index, sorted similarity scores in descending order, and returned the top N most similar anime for a given target anime, and we also experimented with threshold values to control the recommendation list size. Overall, the system gives relevant recommendations for similar-genre anime, and it can be improved further by combining more features (rating, members, type).